In [1]:
# ============================================================
# Notebook    : 06b_case_b_lightgbm_full.ipynb
# Description : Case B — Model B2 LightGBM-Full
#               B1 (demographics) + behavioral/violation history
#               (SPEEDING_VIOLATIONS, DUIS, PAST_ACCIDENTS).
#               Structurally identical to 06a except FEATURE_COLS.
#               Any AUC/F1 difference vs B1 is attributable to
#               the 3 behavioral features alone.
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm scikit-learn pandas numpy


# ============================================================
# 1. Common imports
# ============================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

SEED = 42
np.random.seed(SEED)


# ============================================================
# 2. Load — identical to 06a
# ============================================================
df = pd.read_csv("data/car_insurance_preprocessed.csv")
print(f"[CHECK 1] df shape: {df.shape}")
print(f"[CHECK 1] Positive rate: {df['OUTCOME'].mean()*100:.2f}%")


# ============================================================
# 3. Feature scope — B2 (B1 + behavioral history)
#    ONLY DIFFERENCE FROM 06a: BEHAVIORAL_COLS added
# ============================================================
NUMERIC_COLS = [
    "CREDIT_SCORE", "ANNUAL_MILEAGE", "CHILDREN", "MARRIED",
    "VEHICLE_OWNERSHIP",
]
CAT_COLS = [
    "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION",
    "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE",
]
BINARY_FLAG_COLS = ["CREDIT_SCORE_was_missing", "ANNUAL_MILEAGE_was_missing"]
BEHAVIORAL_COLS  = ["SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]  # NEW vs 06a
FEATURE_COLS = NUMERIC_COLS + CAT_COLS + BINARY_FLAG_COLS + BEHAVIORAL_COLS
LABEL_COL = "OUTCOME"

print(f"\n[CHECK 2] Feature scope (B2 — demographics + behavioral, {len(FEATURE_COLS)} features):")
print(f"  Numeric      : {NUMERIC_COLS}")
print(f"  Categorical  : {CAT_COLS}")
print(f"  Missing flags: {BINARY_FLAG_COLS}")
print(f"  Behavioral   : {BEHAVIORAL_COLS}")


# ============================================================
# 4. Split — IDENTICAL split logic/seed to 06a. Since this is
#    the same dataframe with the same seed and stratify target,
#    the row membership of train/val/test is guaranteed to be
#    identical to 06a (only the feature columns differ).
# ============================================================
X = df[FEATURE_COLS].copy()
y = df[LABEL_COL].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"\n[CHECK 3] Split sizes — train: {len(X_train):,}, "
      f"val: {len(X_val):,}, test: {len(X_test):,}")
print(f"[CHECK 3] Positive rates — "
      f"train: {y_train.mean()*100:.2f}%, "
      f"val: {y_val.mean()*100:.2f}%, "
      f"test: {y_test.mean()*100:.2f}%")


# ============================================================
# 5. Categorical dtype handling
# ============================================================
for col in CAT_COLS:
    X_train[col] = X_train[col].astype("category")
    X_val[col]   = X_val[col].astype("category")
    X_test[col]  = X_test[col].astype("category")

    all_cats = pd.api.types.union_categoricals(
        [X_train[col], X_val[col], X_test[col]]
    ).categories
    X_train[col] = X_train[col].cat.set_categories(all_cats)
    X_val[col]   = X_val[col].cat.set_categories(all_cats)
    X_test[col]  = X_test[col].cat.set_categories(all_cats)

print(f"\n[CHECK 4] X_train shape: {X_train.shape}")


# ============================================================
# 6. Train — identical params/seed to 06a
# ============================================================
POS_RATE = y_train.mean()
SCALE_POS_WEIGHT = (1 - POS_RATE) / POS_RATE
print(f"\n[CHECK 5] scale_pos_weight: {SCALE_POS_WEIGHT:.2f}")

train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=CAT_COLS)
val_set   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=CAT_COLS, reference=train_set)

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "seed": SEED,
    "verbose": -1,
}

print(f"[CHECK 5] Training LightGBM-Full (B2)...")
model = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[train_set, val_set],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=50),
    ],
)
print(f"\n[CHECK 5] Best iteration: {model.best_iteration}")


# ============================================================
# 7. Evaluate
# ============================================================
y_pred_prob = model.predict(X_test, num_iteration=model.best_iteration)
y_pred_cls  = (y_pred_prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_prob)
f1  = f1_score(y_test, y_pred_cls)

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx  = np.argmax(f1_scores[:-1])

print("\n===== Test set evaluation =====")
print(f"Total test rows   : {len(y_test):,}")
print(f"AUC-ROC          : {auc:.4f}")
print(f"F1 @ thr=0.5      : {f1:.4f}")
print(f"Best threshold    : {thresholds[best_idx]:.3f}")
print(f"Best F1           : {f1_scores[best_idx]:.4f}")
print(f"  Precision       : {precisions[best_idx]:.4f}")
print(f"  Recall          : {recalls[best_idx]:.4f}")
print(f"Positive rate in test: {y_test.mean()*100:.2f}%")


# ============================================================
# 8. Feature importance — key diagnostic: how much do the 3
#    behavioral features outrank demographics?
# ============================================================
importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("\n===== Feature importance (gain) =====")
print(importance.to_string(index=False))


# ============================================================
# 9. Save
# ============================================================
model.save_model("data/sequences/case_b_lightgbm_full_model.txt")
np.savez(
    "data/sequences/case_b_lightgbm_full_test_predictions.npz",
    probs=y_pred_prob, labels=y_test.values,
)
print("\nSaved model and predictions.")


# ============================================================
# 10. Summary
# ============================================================
print("\n===== LightGBM-Full (Model B2) Summary =====")
print(f"Feature scope        : {len(FEATURE_COLS)} features (demographics + behavioral)")
print(f"Train / Val / Test   : {len(X_train):,} / {len(X_val):,} / {len(X_test):,}")
print(f"Best iteration       : {model.best_iteration}")
print(f"Test AUC-ROC         : {auc:.4f}")
print(f"Test F1 (best thr)   : {f1_scores[best_idx]:.4f}")
print("===============================================")

[CHECK 1] df shape: (10000, 21)
[CHECK 1] Positive rate: 31.33%

[CHECK 2] Feature scope (B2 — demographics + behavioral, 18 features):
  Numeric      : ['CREDIT_SCORE', 'ANNUAL_MILEAGE', 'CHILDREN', 'MARRIED', 'VEHICLE_OWNERSHIP']
  Categorical  : ['AGE', 'GENDER', 'RACE', 'DRIVING_EXPERIENCE', 'EDUCATION', 'INCOME', 'VEHICLE_YEAR', 'VEHICLE_TYPE']
  Missing flags: ['CREDIT_SCORE_was_missing', 'ANNUAL_MILEAGE_was_missing']
  Behavioral   : ['SPEEDING_VIOLATIONS', 'DUIS', 'PAST_ACCIDENTS']

[CHECK 3] Split sizes — train: 7,000, val: 1,500, test: 1,500
[CHECK 3] Positive rates — train: 31.33%, val: 31.33%, test: 31.33%

[CHECK 4] X_train shape: (7000, 18)

[CHECK 5] scale_pos_weight: 2.19
[CHECK 5] Training LightGBM-Full (B2)...
Training until validation scores don't improve for 30 rounds
[50]	train's auc: 0.922634	val's auc: 0.886602
Early stopping, best iteration is:
[51]	train's auc: 0.922993	val's auc: 0.886842

[CHECK 5] Best iteration: 51

===== Test set evaluation =====
Total tes